In [22]:
import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, RF_PARAM_5G

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 40))
# Data filtering
df = filter_dataframe(
    df=df,
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq', 'sinr', 'rssi', 'rsrp'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [23]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scripts.utils import extract_unique_npcis
import pandas as pd


def train_kmeans(df: pd.DataFrame, n_clusters: int, random_state: int, rf_param, unique_npcis):
    df_features, _ = create_point_matrix(df, unique_npcis, rf_param)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(df_features)

    # Calculate inertia
    inertia = kmeans.inertia_

    # Calculate silhouette score
    silhouette_avg = silhouette_score(df_features, cluster_labels)

    # Calculate Davies-Bouldin score
    davies_bouldin = davies_bouldin_score(df_features, cluster_labels)

    return (inertia, silhouette_avg, davies_bouldin)


print('KMEANS PERFORMANCE')

# Example usage
unique_npcis = extract_unique_npcis(df['measurements_matrix'])
rf_param = RF_PARAM_5G.RSRQ

data = []
for n_clusters in range(2, 21):
    print(f'Testing with k={n_clusters}')
    r = train_kmeans(df, n_clusters=n_clusters, random_state=42, rf_param=rf_param, unique_npcis=unique_npcis)

    data.append((n_clusters,) + r)

# Print the data to verify its structure
print(data)

# Define columns
cols = ['K', 'Inertia', 'Silhouette Score', 'Davies-Bouldin Score']

# Create DataFrame
res_df = pd.DataFrame(data, columns=cols)

# Print the resulting DataFrame
print(res_df)

KMEANS PERFORMANCE
Testing with k=2
Testing with k=3
Testing with k=4
Testing with k=5
Testing with k=6
Testing with k=7
Testing with k=8
Testing with k=9
Testing with k=10
Testing with k=11
Testing with k=12
Testing with k=13
Testing with k=14
Testing with k=15
Testing with k=16
Testing with k=17
Testing with k=18
Testing with k=19
Testing with k=20
[(2, 211492600.15308893, np.float64(0.28869748535986617), np.float64(1.4366212336119695)), (3, 178757162.98865718, np.float64(0.2518594982947036), np.float64(1.6835664533688812)), (4, 163853449.4856969, np.float64(0.22581573395776008), np.float64(1.7924031949845212)), (5, 153711947.1745618, np.float64(0.20670286927783807), np.float64(1.9125114727294903)), (6, 140358541.8215986, np.float64(0.21869274133977837), np.float64(1.6965106100025569)), (7, 136245423.1703832, np.float64(0.16107817412122113), np.float64(1.9965495543401903)), (8, 125155612.24601984, np.float64(0.1932382179983871), np.float64(1.7306180664786672)), (9, 120534689.03501058

In [24]:
res_df.to_csv('kmeans.csv')
res_df

,K,Inertia,Silhouette Score,Davies-Bouldin Score
0,2,2.114926e+08,0.288697,1.436621
1,3,1.787572e+08,0.251859,1.683566
2,4,1.638534e+08,0.225816,1.792403
3,5,1.537119e+08,0.206703,1.912511
4,6,1.403585e+08,0.218693,1.696511
5,7,1.362454e+08,0.161078,1.996550
6,8,1.251556e+08,0.193238,1.730618
7,9,1.205347e+08,0.179760,1.797044
8,10,1.179782e+08,0.178871,1.850509
9,11,1.126917e+08,0.188032,1.726995


In [2]:
#df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 42)

unique_npcis = extract_unique_npcis(df['measurements_matrix'])
rf_param = RF_PARAM_5G.RSRQ
n_clusters = 5
random_seed = 42
n_runs = 10

In [ ]:
from scripts.matrix_operations import create_point_matrix
from scripts.utils import dataset_tp_rp_split
from scripts.clustering import train_random_forest
import concurrent.futures
import numpy as np


def train_kmeans(df: pd.DataFrame, n_clusters: int, random_state: int):
    df_features, _ = create_point_matrix(df, unique_npcis, rf_param)

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(df_features)
    return kmeans, cluster_labels


def evaluate_kmeans(df_features, cluster_labels):
    # Calculate inertia
    inertia = KMeans.inertia_

    # Calculate silhouette score
    silhouette_avg = silhouette_score(df_features, cluster_labels)

    # Calculate Davies-Bouldin score
    davies_bouldin = davies_bouldin_score(df_features, cluster_labels)

    return {
        'Inertia': inertia,
        'Silhouette Score': silhouette_avg,
        'Davies-Bouldin Score': davies_bouldin
    }


def process_single_run(df, c, i, random_seeds, unique_npcis, rf_param):
    """Process a single run for a specific cluster size and random seed"""
    random_seed = random_seeds[i]

    # cluster the points
    kmeans, cluster_labels = train_kmeans(df, c, random_seed)
    df_with_clusters = df.copy()
    df_with_clusters['cluster'] = cluster_labels

    # split the dataset
    df_tp, df_rp = dataset_tp_rp_split(df_with_clusters, 0.3, random_seed)

    # train the random forest
    rf = train_random_forest(df_rp, unique_npcis, rf_param, n_estimators=100, random_state=random_seed)

    # predict the cluster
    tp_features, _ = create_point_matrix(df_tp, unique_npcis, rf_param)
    df_tp['predicted'] = rf.predict(tp_features)

    # get the accuracy
    hits = df_tp[df_tp['cluster'] == df_tp['predicted']].shape[0]
    total = df_tp.shape[0]

    # return result
    accuracy = hits / total * 100
    print(f'\rCluster {c} - Run {i + 1}/{n_runs} - Accuracy: {accuracy:.2f}%', end='')
    return accuracy


def process_cluster_size(df, c, n_runs, random_seeds, unique_npcis, rf_param):
    """Process all runs for a specific cluster size using threads"""
    results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Submit all runs for this cluster size to the thread pool
        future_to_run = {
            executor.submit(
                process_single_run,
                df, c, i, random_seeds, unique_npcis, rf_param
            ): i for i in range(n_runs)
        }

        # Collect results as they complete
        for future in concurrent.futures.as_completed(future_to_run):
            run_index = future_to_run[future]
            try:
                accuracy = future.result()
                results.append(accuracy)
            except Exception as exc:
                print(f'\nRun {run_index} for cluster {c} generated an exception: {exc}')

    print(f'\nCluster {c} - Average accuracy: {np.mean(results):.2f}%')
    return results


# # Main execution
# cluster_range = range(1, 10)
# n_runs = len(random_seeds)  # Assuming random_seeds is defined elsewhere
#
# # Dictionary to store results
# all_results = {c: [] for c in cluster_range}
#
# # Process each cluster size (this could also be parallelized if needed)
# for c in cluster_range:
#     print(f"Processing cluster size {c}...")
#     all_results[c] = process_cluster_size(df, c, n_runs, random_seeds, unique_npcis, rf_param)
#
# # Now all_results contains the accuracy scores for each cluster size and run


kmeans, cluster_labels = train_kmeans(df, n_clusters=3, random_state=42)
performance_metrics = evaluate_kmeans(df_features, cluster_labels)

print(performance_metrics)

Processing cluster size 1...
Cluster 1 - Run 1000/1000 - Accuracy: 100.00%
Cluster 1 - Average accuracy: 100.00%
Processing cluster size 2...
Cluster 2 - Run 1000/1000 - Accuracy: 99.59%
Cluster 2 - Average accuracy: 99.56%
Processing cluster size 3...
Cluster 3 - Run 600/1000 - Accuracy: 99.49%